# Verifying the music-dsp feedback (R. Bristow-Johnson, June 2026)

After SampleRateTap was announced on the [music-dsp list](https://listserv.cuit.columbia.edu/scripts/wa.exe?A2=2607A&L=MUSIC-DSP&D=0&P=682445),
Robert Bristow-Johnson replied with detailed feedback containing several quantitative
claims about polyphase interpolator design. This notebook checks each claim against
this library's actual filters — the design math below is the same formula-for-formula
port of `include/srt/detail/kaiser.h` used by `scripts/book_figures.py`, and every
result is pinned by an assertion so regressions in either the claims or the code fail loudly.

**The claims under test:**

1. With linear inter-phase interpolation, folded-image S/N is ~108 dB at L=256 and
   120+ dB at L=512, improving 12 dB per doubling of L (6 dB for drop-sample).
2. "That's a lotta taps" — T=32 taps per phase suffices instead of our T=48.
3. Putting zeros at integer multiples of fs (convolving the prototype with a
   one-sample rect) usefully deepens the low-order images.
4. Two dot products + one output interpolation beats blending coefficients then one dot.
5. The phase table should have L+1 rows spanning delays T/2−1 … T/2 samples, with the
   first prototype tap zero by symmetry.

Method note: claims 1–3 are measured by *actually fractionally resampling* a sine
through the table (ratio 1 + 200 ppm) and least-squares-fitting the expected tone;
the residual is everything the interpolator got wrong. 19.5 kHz is used as the
near-worst case (interpolation error scales with signal frequency); 997 Hz shows
the mid-band behaviour our headline numbers are measured at.

In [1]:
import sys
import numpy as np

sys.path.insert(0, "../scripts")
from book_figures import design_prototype, kaiser_beta  # kaiser.h, ported verbatim

FS = 48000.0

def proto(L, T, pass_hz, stop_hz, atten):
    return design_prototype(L, T, (pass_hz + stop_hz) / FS, kaiser_beta(atten))

def resample_snr(L, T, pass_hz, stop_hz, atten, f0, eps=2e-4, N=150_000, mode="linear"):
    """Fractionally resample a sine using the (L+1)-row table with linear
    coefficient interpolation (== linear interpolation on the oversampled
    prototype grid), LS-fit the expected tone, return residual SNR in dB."""
    h = proto(L, T, pass_hz, stop_hz, atten)      # length L*T, grid 1/L, DC gain L
    n = np.arange(N, dtype=float)
    pos = n * (1.0 + eps) + T + 2
    idx = np.floor(pos).astype(int)
    x = np.sin(2 * np.pi * f0 * np.arange(idx.max() + T + 4) / FS)
    m = idx[:, None] + np.arange(-(T // 2) + 1, T // 2 + 1)[None, :]
    u = pos[:, None] - m                           # kernel argument in (-T/2, T/2]
    g = u * L + (L * T - 1) / 2.0                  # oversampled-prototype index
    if mode == "linear":
        g0 = np.floor(g).astype(int); fr = g - g0
        g0c = np.clip(g0, 0, L * T - 1); g1c = np.clip(g0 + 1, 0, L * T - 1)
        P = (1 - fr) * h[g0c] * (g0 >= 0) + fr * h[g1c] * (g0 + 1 <= L * T - 1)
    else:                                          # drop-sample / nearest phase row
        gn = np.rint(g).astype(int)
        P = h[np.clip(gn, 0, L * T - 1)] * ((gn >= 0) & (gn <= L * T - 1))
    y = (x[m] * P).sum(axis=1)
    t = pos / FS
    A = np.column_stack([np.sin(2 * np.pi * f0 * t), np.cos(2 * np.pi * f0 * t)])
    y, A = y[N // 10:], A[N // 10:]                # drop the fill transient
    c, *_ = np.linalg.lstsq(A, y, rcond=None)
    r = y - A @ c
    return 10 * np.log10((A @ c).var() / r.var())

## Claim 1 — the L budget: ~108 dB at 256, 120+ at 512, 12 dB/doubling

A 140 dB / T=80 FIR isolates the phase-interpolation error (the FIR's own stopband
sits well below every floor we are trying to measure).

In [2]:
snr = {}
prev = None
for L in (128, 256, 512, 1024):
    snr[L] = resample_snr(L, 80, 20000, 26000, 140, 19500)
    step = "" if prev is None else f"   (+{snr[L]-prev:.1f} dB per doubling)"
    print(f"L={L:5}: {snr[L]:6.1f} dB{step}")
    prev = snr[L]

drop = {L: resample_snr(L, 80, 20000, 26000, 140, 19500, mode="nearest") for L in (256, 512)}
print(f"drop-sample: L=256 {drop[256]:.1f} dB, L=512 {drop[512]:.1f} dB "
      f"(+{drop[512]-drop[256]:.1f} dB per doubling)")

assert 107.0 < snr[256] < 110.0, "RBJ's ~108 dB at L=256"
assert 119.0 < snr[512] < 123.0, "RBJ's 120+ dB at L=512"
assert 11.0 < snr[512] - snr[256] < 13.0, "12 dB per doubling (linear)"
assert 5.0 < drop[512] - drop[256] < 7.0, "6 dB per doubling (drop-sample)"
print("\nCONFIRMED: all four figures match RBJ's rules of thumb.")

L=  128:   96.6 dB


L=  256:  108.6 dB   (+12.0 dB per doubling)


L=  512:  120.7 dB   (+12.0 dB per doubling)


L= 1024:  132.5 dB   (+11.8 dB per doubling)


drop-sample: L=256 50.8 dB, L=512 56.8 dB (+6.0 dB per doubling)

CONFIRMED: all four figures match RBJ's rules of thumb.


**Verdict: confirmed, remarkably precisely.** His ~108/120+/12-per-doubling figures
reproduce to within half a decibel on our own prototype. Note the frequency dependence:
this floor is for a 19.5 kHz tone; the same L=256 table measures ~144 dB at 997 Hz,
which is why the library's mid-band headline measurements (126–135 dB through the full
converter) coexist with a ~108 dB near-Nyquist interpolation floor — and why
`test_asrc_quality`'s 19.5 kHz threshold sits at 100 dB rather than 110.

## Claim 2 — "I think you can get away with T=32"

T is not spent on phase resolution — it is spent on the FIR's own transition width and
stopband depth. Pair T=32 (whose achievable Kaiser design at our band edges is the
96 dB `fast` preset) with as many phases as you like, and the FIR stopband dominates:

In [3]:
cases = [
    ("fast        L=128 T=32", 128, 32, 18000, 30000,  96),
    ("balanced    L=256 T=48", 256, 48, 20000, 28000, 120),
    ("balanced @ L=512      ", 512, 48, 20000, 28000, 120),
    ("RBJ's T=32 @ L=512    ", 512, 32, 18000, 30000,  96),
    ("transparent L=512 T=80", 512, 80, 20000, 26000, 140),
]
res = {}
for name, L, T, pb, sb, att in cases:
    hi = resample_snr(L, T, pb, sb, att, 19500)
    lo = resample_snr(L, T, pb, sb, att, 997)
    res[name.strip()] = (hi, lo)
    print(f"{name}:  19.5 kHz {hi:6.1f} dB    997 Hz {lo:6.1f} dB")

assert res["RBJ's T=32 @ L=512"][0] < 90, "T=32 tops out near the 96 dB design floor"
assert res["balanced    L=256 T=48"][0] > 105
print("\nREFUTED at the 120 dB spec: quadrupling L cannot rescue a 96 dB-class FIR.")

fast        L=128 T=32:  19.5 kHz   83.9 dB    997 Hz  110.7 dB


balanced    L=256 T=48:  19.5 kHz  108.5 dB    997 Hz  144.3 dB


balanced @ L=512      :  19.5 kHz  118.9 dB    997 Hz  144.3 dB


RBJ's T=32 @ L=512    :  19.5 kHz   84.2 dB    997 Hz  110.7 dB


transparent L=512 T=80:  19.5 kHz  120.7 dB    997 Hz  153.5 dB

REFUTED at the 120 dB spec: quadrupling L cannot rescue a 96 dB-class FIR.


**Verdict: refuted at our specification.** T=32 with L=512 delivers 84 dB at
19.5 kHz — the FIR stopband, not the phase count, is the binding constraint. The
harris length estimate says the same thing analytically: 120 dB across an 8 kHz
transition needs T ≈ 47 (Kaiser) or ≈ 44 (equiripple/Parks–McClellan). T=48 *is*
the 120 dB spec; T=32 *is* our `fast` preset, offered for the 96 dB tier. (RBJ's
own designs pair T=32 with optimal PM/LS prototypes and a less demanding stopband
target — in that budget his claim is fair.)

## Claim 3 — zeros at multiples of fs (convolve the prototype with a rect)

Real technique, real benefit at the low-order images — and a real cost RBJ's optimal
designs compensate but a naive rect-on-Kaiser does not: sinc droop across the passband.

In [4]:
L, T = 256, 48
h1 = proto(L, T, 20000, 28000, 120)
h2 = np.convolve(h1, np.ones(L) / L)     # zeros at k*fs; T -> T+1 taps per phase
report = {}
for tag, hh in (("as shipped", h1), ("rect-convolved", h2)):
    nfft = 1 << 21
    H = np.abs(np.fft.rfft(hh, nfft)) / L
    f = np.arange(H.size) * (L * FS) / nfft
    db = 20 * np.log10(np.maximum(H, 1e-16))
    band = lambda lo, hi: db[(f >= lo) & (f <= hi)].max()
    report[tag] = (band(19990, 20010), band(FS - 20000, FS + 20000), band(2*FS - 20000, 2*FS + 20000))
    print(f"{tag:15}: droop@20k {report[tag][0]:6.2f} dB | "
          f"worst image fs±20k {report[tag][1]:7.1f} dB | 2fs±20k {report[tag][2]:7.1f} dB")

droop = report["rect-convolved"][0]
img_gain = report["as shipped"][1] - report["rect-convolved"][1]
assert droop < -2.0, "sinc droop at the passband edge is material"
assert img_gain > 4.0, "the image-band benefit is also real"
print(f"\nBOTH SIDES REAL: {img_gain:.1f} dB deeper first image, {droop:.2f} dB droop at 20 kHz.")

as shipped     : droop@20k   0.00 dB | worst image fs±20k  -119.1 dB | 2fs±20k  -148.0 dB
rect-convolved : droop@20k  -2.64 dB | worst image fs±20k  -124.7 dB | 2fs±20k  -162.3 dB

BOTH SIDES REAL: 5.6 dB deeper first image, -2.64 dB droop at 20 kHz.


**Verdict: correct but not free.** ~6 dB deeper first-image rejection and ~14 dB at
2·fs, in exchange for **−2.6 dB at 20 kHz**, which violates this library's ±0.01 dB
passband spec by two orders of magnitude. Adopting the idea properly means an optimal
(PM/LS) design with the sinc droop pre-compensated in the passband target — a genuine
future experiment, not a drop-in change. (RBJ's MATLAB designs do exactly this.)

## Claim 4 — "two dot products and one linear interpolation would be quicker"

An operation-count question, and for the multichannel case a measured one. Per output
frame with T taps and N channels:

| approach | per-frame ops |
|---|---|
| blend row once, dot per channel (ours) | T blend + N·T MAC ≈ (N+1)·T |
| two dots per channel, lerp outputs (RBJ) | 2·N·T MAC + N lerp ≈ 2·N·T |

Mono: a wash (2T either way — his form even saves the scratch-row traffic; on
Cortex-M33-class targets our Q15 path already prefers fused/dual-MAC forms).
N ≥ 2: the blend amortizes across channels and ours wins by (N−1)·T ops — which is
not hypothetical: the C1 campaign measured **−36 % stereo and −52 % 8-channel**
wall-clock from exactly this restructuring (`docs/PERFORMANCE.md`, C1), and the C6
channel-parallel path (frame-major, one blended row broadcast across channel lanes)
requires the shared blended row to exist at all.

## Claim 5 — table conventions (L+1 rows, delay span, zero first tap)

No disagreement to test — just verify we implement the same convention he describes.

In [5]:
h = proto(256, 48, 20000, 28000, 120)
gd = (256 * 48 - 1) / (2 * 256)
print(f"h[0] = {h[0]:.3e}   (RBJ's convention: exactly 0 by symmetry)")
print(f"group delay = (L·T−1)/(2L) = {gd:.4f} samples = T/2 − 1/(2L)")
print("rows stored: L+1 (row L = row 0 advanced one input sample) — see PolyphaseFilterBank")
assert abs(h[0]) < 1e-7
assert abs(gd - (24 - 1/512)) < 1e-9
print("\nSAME DESIGN: delay spans T/2−1 … T/2 across the L+1 rows, first tap ≈ 0.")

h[0] = -3.331e-09   (RBJ's convention: exactly 0 by symmetry)
group delay = (L·T−1)/(2L) = 23.9980 samples = T/2 − 1/(2L)
rows stored: L+1 (row L = row 0 advanced one input sample) — see PolyphaseFilterBank

SAME DESIGN: delay spans T/2−1 … T/2 across the L+1 rows, first tap ≈ 0.


## Conclusions

| # | Claim | Verdict |
|---|---|---|
| 1 | ~108 dB @ L=256, 120+ @ L=512, 12 dB/doubling (6 for drop-sample) | **Confirmed** to ±0.5 dB |
| 2 | T=32 suffices | **Refuted at the 120 dB spec** — FIR stopband dominates; T=32 is the 96 dB `fast` tier |
| 3 | Zeros at k·fs via rect convolution | **Real benefit, real cost** — needs droop-precompensated optimal design |
| 4 | Two dots + output lerp | **Mono: wash. Multichannel: ours measured faster** (C1: −36 %/−52 %) |
| 5 | L+1 rows, T/2−1…T/2 delay span, zero first tap | **Same convention**, verified |

Also raised in the thread: per-sample timestamping as an alternative servo observable
(interesting for acquisition speed; our occupancy+μ observable is already continuous and
whole-sample slips are bit-continuous by the extra-row property — `AsrcLock.WholeSampleSlipsAreGlitchFree`),
and Q32.32 absolute indexing vs our Q0.64 deviation-only accumulator (equivalent;
near-unity lets the integer advance carry the 1).

---

# Experiments

The two follow-ups promised in the list reply, run to first results.

## Experiment 1 — zeros at k·fs with droop pre-compensation (claim 3, done properly)

Same idea as RBJ's optimal designs: pre-emphasize the passband target by
1/sinc(f/fs) *before* windowing, then convolve with the one-sample rect, so the
composite has the k·fs zeros **and** a flat passband. Compared at **equal total
taps per phase** (47+1 vs the shipped 48).

In [6]:
def freq_sample_proto(L, T, pass_hz, stop_hz, beta, precomp=True):
    """Kaiser-windowed frequency-sampling design on the L*fs grid; passband
    target optionally pre-emphasized by 1/sinc(f/fs) so a following rect
    convolution lands flat."""
    from book_figures import bessel_i0
    n = L * T
    nfft = 1 << 20
    f = np.fft.rfftfreq(nfft, d=1.0 / (L * FS))
    cutoff = 0.5 * (pass_hz + stop_hz)
    D = (f <= cutoff).astype(float)
    if precomp:
        s = np.sinc(f / FS)
        D = np.where(f <= cutoff, D / np.maximum(np.abs(s), 1e-3), 0.0)
    d = np.roll(np.fft.irfft(D), n // 2)[:n]
    u = (np.arange(n) - 0.5 * (n - 1)) / (0.5 * (n - 1))
    w = bessel_i0(beta * np.sqrt(np.maximum(0, 1 - u * u))) / bessel_i0(beta)
    h = d * w
    return h * (L / h.sum())

def resample_snr_h(h, L, T, f0, eps=2e-4, N=150_000):
    """resample_snr for an explicit prototype (length may exceed L*T)."""
    n = np.arange(N, dtype=float)
    pos = n * (1.0 + eps) + T + 2
    idx = np.floor(pos).astype(int)
    x = np.sin(2 * np.pi * f0 * np.arange(idx.max() + T + 4) / FS)
    m = idx[:, None] + np.arange(-(T // 2) + 1, T // 2 + 1)[None, :]
    g = (pos[:, None] - m) * L + (len(h) - 1) / 2.0
    g0 = np.floor(g).astype(int); fr = g - g0; nmax = len(h) - 1
    P = ((1 - fr) * h[np.clip(g0, 0, nmax)] * (g0 >= 0)
         + fr * h[np.clip(g0 + 1, 0, nmax)] * (g0 + 1 <= nmax))
    y = (x[m] * P).sum(axis=1)
    t = pos / FS
    A = np.column_stack([np.sin(2 * np.pi * f0 * t), np.cos(2 * np.pi * f0 * t)])
    y, A = y[N // 10:], A[N // 10:]
    c, *_ = np.linalg.lstsq(A, y, rcond=None)
    r = y - A @ c
    return 10 * np.log10((A @ c).var() / r.var())

def spec_report(h, L):
    nfft = 1 << 21
    H = np.abs(np.fft.rfft(h, nfft)) / L
    f = np.arange(H.size) * (L * FS) / nfft
    db = 20 * np.log10(np.maximum(H, 1e-16))
    pb = db[(f >= 20) & (f <= 20000)]
    band = lambda lo, hi: db[(f >= lo) & (f <= hi)].max()
    return pb.min(), pb.max(), band(FS - 20000, FS + 20000), band(2 * FS - 20000, 2 * FS + 20000)

L = 256
beta = kaiser_beta(120.0)
h_base = proto(L, 48, 20000, 28000, 120)
h_cand = np.convolve(freq_sample_proto(L, 47, 20000, 28000, beta), np.ones(L) / L)
h_cand *= L / h_cand.sum()

rows = {}
for name, h, T in (("Kaiser T=48 (shipped)", h_base, 48),
                   ("precomp+rect (47+1 taps)", h_cand, 48)):
    mn, mx, i1, i2 = spec_report(h, L)
    hi = resample_snr_h(h, L, T, 19500)
    lo = resample_snr_h(h, L, T, 997)
    rows[name] = (mn, mx, i1, i2, hi, lo)
    print(f"{name:26}: passband [{mn:+7.3f},{mx:+7.3f}] dB | image fs±20k {i1:7.1f} | "
          f"SNR 19.5k {hi:6.1f} | 997 Hz {lo:6.1f} dB")

base, cand = rows["Kaiser T=48 (shipped)"], rows["precomp+rect (47+1 taps)"]
assert cand[1] < 0.010 and cand[0] > -0.010, "passband within ±0.01 dB after precomp"
assert cand[5] - base[5] > 10, "double-digit dB win for mid/low-band folded images"
assert abs(cand[4] - base[4]) < 1.0, "near-Nyquist unchanged (interpolation floor dominates)"
print(f"\nWIN at equal cost: +{cand[5]-base[5]:.1f} dB @ 997 Hz, passband flat to "
      f"{cand[1]:+.3f} dB, near-Nyquist unchanged.")

Kaiser T=48 (shipped)     : passband [ -0.000, +0.000] dB | image fs±20k  -119.1 | SNR 19.5k  108.5 | 997 Hz  144.3 dB


precomp+rect (47+1 taps)  : passband [ -0.000, +0.009] dB | image fs±20k  -121.8 | SNR 19.5k  108.4 | 997 Hz  158.0 dB

WIN at equal cost: +13.8 dB @ 997 Hz, passband flat to +0.009 dB, near-Nyquist unchanged.


**Result: RBJ's suggestion survives the passband spec once pre-compensated, and wins
double-digit decibels for low/mid-band program material at equal tap count.** The ripple
margin (+0.009 dB against a ±0.01 dB spec) is thin — a properly weighted LS design should
buy margin, which is the productionization step if this graduates into a `FilterSpec`
option. Near-Nyquist tones are unchanged because the inter-phase interpolation floor
(claim 1) dominates there.

## Experiment 2 — mono kernel ordering: blend-then-dot vs two-dots-then-lerp (claim 4)

Multichannel was already settled by the C1 record; the open question was mono, where the
op counts tie at ≈ 2T. The cell below writes, compiles (`-O3 -march=native`), and runs an
interleaved same-machine A/B with the library's accumulation contract (float data, ordered
double accumulation — `SampleTraits<float>`). Wall-clock, one machine, not a CI gate.

In [7]:
import subprocess, tempfile, textwrap, os, sys
src = textwrap.dedent(r'''
#include <chrono>
#include <cstdio>
#include <random>
#include <vector>
constexpr int T = 48; constexpr int N = 1 << 14; constexpr int REPS = 2000;
int main() {
    std::mt19937 rng(42); std::uniform_real_distribution<float> d(-1.f, 1.f);
    std::vector<float> c0(N*T), c1(N*T), hist(N*T), fr(N);
    for (auto v : {&c0, &c1, &hist}) for (auto& x : *v) x = d(rng);
    for (auto& x : fr) x = 0.5f * (d(rng) + 1.f);
    auto benchA = [&] { double acc = 0; float row[T];
        for (int i = 0; i < N; ++i) {
            const float *a = &c0[i*T], *b = &c1[i*T], *h = &hist[i*T]; const float f = fr[i];
            for (int t = 0; t < T; ++t) row[t] = a[t] + f * (b[t] - a[t]);
            double s = 0; for (int t = 0; t < T; ++t) s += (double)h[t] * row[t];
            acc += s; } return acc; };
    auto benchB = [&] { double acc = 0;
        for (int i = 0; i < N; ++i) {
            const float *a = &c0[i*T], *b = &c1[i*T], *h = &hist[i*T]; const float f = fr[i];
            double s0 = 0, s1 = 0;
            for (int t = 0; t < T; ++t) { s0 += (double)h[t] * a[t]; s1 += (double)h[t] * b[t]; }
            acc += s0 + f * (s1 - s0); } return acc; };
    double sum = 0, tA = 0, tB = 0; using clk = std::chrono::steady_clock;
    for (int r = 0; r < REPS; ++r) {
        auto t0 = clk::now(); sum += benchA();
        auto t1 = clk::now(); sum += benchB();
        auto t2 = clk::now();
        tA += std::chrono::duration<double>(t1 - t0).count();
        tB += std::chrono::duration<double>(t2 - t1).count(); }
    const double per = 1e9 / ((double)N * REPS);
    std::printf("blend+dot : %.2f ns/output\ntwo dots  : %.2f ns/output\nratio B/A : %.3f (checksum %g)\n",
                tA * per, tB * per, tB / tA, sum);
    return 0; }
''')
with tempfile.TemporaryDirectory() as tmp:
    cpp = os.path.join(tmp, "kernel_order.cpp"); exe = os.path.join(tmp, "kernel_order")
    open(cpp, "w").write(src)
    subprocess.run(["g++", "-O3", "-march=native", "-std=c++20", cpp, "-o", exe], check=True)
    out = subprocess.run([exe], check=True, capture_output=True, text=True).stdout
print(out)
ratio = float(out.split("ratio B/A :")[1].split("(")[0])
assert 0.3 < ratio < 3.0, "sanity only: the DIRECTION is machine-dependent (see below)"
print(f"on THIS host, two-dots-then-lerp runs at {ratio:.2f}x the blend+dot time "
      f"({'slower' if ratio > 1 else 'faster'})")

blend+dot : 78.19 ns/output
two dots  : 50.66 ns/output
ratio B/A : 0.648 (checksum 510236)

on THIS host, two-dots-then-lerp runs at 0.65x the blend+dot time (faster)


**Result: machine-dependent — and that is the finding.** Two hosts, freshly
compiled `-O3 -march=native` on each, interleaved A/B:

| host | blend+dot | two dots | verdict |
|---|---|---|---|
| ~5 GHz desktop-class, AVX2+FMA | 39 ns | 51 ns | swap **1.3× slower** |
| 2.8 GHz Xeon, AVX-512 | 72 ns | 50 ns | swap **1.4× faster** |

The mechanism: under this library's quality contract each dot accumulates in
double, in a fixed order — a serial FP-add dependency chain (~T × add-latency,
which is why blend+dot tracks the host's scalar latency×clock almost exactly:
48 × 4 cyc ≈ 69 ns at 2.8 GHz). RBJ's two-dots form pays for a second dot but
gets two **independent** chains that overlap in the pipeline, while its output
lerp is trivial; the blend it avoids is vectorizable float work whose cost
varies by vector ISA. Fast scalar host → blend+dot wins; slow-clock
wide-vector host → two-dots wins.

Consequences for the library: (1) the multichannel answer is unchanged — the
shared blended row's (N+1)·T vs 2·N·T advantage and the measured C1 −36 %/−52 %
stand, and the C6 path requires the shared row; (2) the mono ordering is a
per-target empirical question that would need the instruction-count ratchet on
each embedded target to settle, not a host wall-clock; (3) the two orderings
are *not* bit-identical (lerp-of-dots vs dot-of-lerped-row round differently),
so under this project's rules a swap is an algorithm change needing quality
evidence, not a free optimization. Recorded as an open, honest draw — RBJ's
instinct is right on at least one class of machine.